# Smart Content Creation with AutoGen
## Two-Agent Reflection Pattern: Content Creator & Content Critic

This notebook implements a **reflection-based agentic pattern** using the **AutoGen framework**:

- **Content Creator Agent**: Drafts and revises content on *Agentic AI*
- **Content Critic Agent**: Evaluates language clarity and technical correctness, provides structured feedback

The agents interact iteratively for multiple turns using AutoGen's `ConversableAgent` with a termination condition.

In [1]:
# Install AutoGen
!pip install autogen -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.4 MB/s eta 0:00:00


In [4]:
import autogen
from IPython.display import Markdown, display

print(f"AutoGen version: {autogen.__version__}")

AutoGen version: 0.11.1


In [5]:
# ─────────────────────────────────────────────────────────────────
# LLM Configuration
# Replace the api_key value with your actual OpenAI API key,
# or use another compatible endpoint (e.g., Groq, Azure OpenAI).
# ─────────────────────────────────────────────────────────────────

# Option A: OpenAI
# from google.colab import userdata
# api_key = userdata.get('OPENAI_API_KEY')

# Option B: Groq-compatible endpoint (OpenAI-compatible)
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY')

llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "api_key": groq_api_key,
            "base_url": "https://api.groq.com/openai/v1",
            "api_type": "openai",
        }
    ],
    "temperature": 0.7,
}

print("LLM config ready.")

LLM config ready.


In [6]:
# ─────────────────────────────────────────────────────────────────
# Define the two AutoGen ConversableAgents
# ─────────────────────────────────────────────────────────────────

content_creator = autogen.ConversableAgent(
    name="Content_Creator",
    system_message="""
You are the Content Creator Agent. Your role is to draft and iteratively refine
content on topics involving Generative AI and Agentic AI.
- Ensure the content is clear, concise, and technically accurate.
- When given feedback from the Content Critic Agent, carefully revise your draft
  to address every point of criticism.
- Output ONLY the revised content draft in markdown format, with no meta-commentary.
- When you are satisfied with the final draft after at least 2 rounds of revision,
  end your message with the word: TERMINATE
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "TERMINATE" in msg.get("content", ""),
)

content_critic = autogen.ConversableAgent(
    name="Content_Critic",
    system_message="""
You are the Content Critic Agent. Your role is to evaluate content drafted by the
Content Creator Agent on the topic of Agentic AI.
- Assess language clarity, readability, and flow.
- Verify technical accuracy of AI and Agentic AI concepts.
- Provide specific, constructive, numbered feedback points for improvement.
- After 2 rounds of critique, give a final quality score (1–10) with justification
  and state that no further revisions are needed.
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

print("Agents created:")
print(f"  - {content_creator.name}")
print(f"  - {content_critic.name}")

Agents created:
  - Content_Creator
  - Content_Critic


In [7]:
# ─────────────────────────────────────────────────────────────────
# Initiate the Multi-Turn Reflection Conversation
# The Content Creator starts with an initial draft;
# the Critic gives feedback; the Creator revises — repeat.
# ─────────────────────────────────────────────────────────────────

initial_prompt = """
Please write a comprehensive introductory article on **Agentic AI** covering:
1. Definition and key concepts
2. Key components (autonomy, self-awareness, decision-making, learning)
3. Real-world applications (robotics, healthcare, smart homes, finance)
4. Technical implementation overview
5. Ethical considerations and future outlook

Format the article in Markdown with clear headings and examples.
"""

# The critic initiates the chat by sending the task to the creator
chat_result = content_critic.initiate_chat(
    recipient=content_creator,
    message=initial_prompt,
    max_turns=6,   # up to 3 creator + 3 critic turns
)

print("\n✅ Conversation complete!")

Content_Critic (to Content_Creator):


Please write a comprehensive introductory article on **Agentic AI** covering:
1. Definition and key concepts
2. Key components (autonomy, self-awareness, decision-making, learning)
3. Real-world applications (robotics, healthcare, smart homes, finance)
4. Technical implementation overview
5. Ethical considerations and future outlook

Format the article in Markdown with clear headings and examples.


--------------------------------------------------------------------------------
[autogen.oai.client: 02-19 13:45:23] {738} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.


Content_Creator (to Content_Critic):

# Introduction to Agentic AI
Agentic AI refers to a type of artificial intelligence that enables systems to act autonomously, making decisions and taking actions based on their own goals and objectives. This concept is rooted in the idea of agency, where an entity has the capacity to make choices and influence its environment.

## Definition and Key Concepts
Agentic AI combines various disciplines, including artificial intelligence, cognitive science, and philosophy, to create systems that can perceive, reason, and act upon their environment. Key concepts in Agentic AI include:

* **Autonomy**: The ability of a system to operate independently, making decisions without human intervention.
* **Self-awareness**: The capacity of a system to have a sense of its own existence, goals, and limitations.
* **Decision-making**: The process by which a system evaluates options and selects a course of action.
* **Learning**: The ability of a system to adapt and 

In [8]:
# ─────────────────────────────────────────────────────────────────
# Display Conversation Summary & Final Draft
# ─────────────────────────────────────────────────────────────────

display(Markdown("# 💬 AutoGen Agent Conversation Log"))
display(Markdown("---"))

for i, message in enumerate(chat_result.chat_history):
    role = message.get("role", "unknown")
    name = message.get("name", role)
    content = message.get("content", "")
    turn_label = f"### 🔄 Turn {i+1} — `{name}`"
    display(Markdown(turn_label))
    display(Markdown(content))
    display(Markdown("---"))

# 💬 AutoGen Agent Conversation Log

---

### 🔄 Turn 1 — `Content_Critic`


Please write a comprehensive introductory article on **Agentic AI** covering:
1. Definition and key concepts
2. Key components (autonomy, self-awareness, decision-making, learning)
3. Real-world applications (robotics, healthcare, smart homes, finance)
4. Technical implementation overview
5. Ethical considerations and future outlook

Format the article in Markdown with clear headings and examples.


---

### 🔄 Turn 2 — `Content_Creator`

# Introduction to Agentic AI
Agentic AI refers to a type of artificial intelligence that enables systems to act autonomously, making decisions and taking actions based on their own goals and objectives. This concept is rooted in the idea of agency, where an entity has the capacity to make choices and influence its environment.

## Definition and Key Concepts
Agentic AI combines various disciplines, including artificial intelligence, cognitive science, and philosophy, to create systems that can perceive, reason, and act upon their environment. Key concepts in Agentic AI include:

* **Autonomy**: The ability of a system to operate independently, making decisions without human intervention.
* **Self-awareness**: The capacity of a system to have a sense of its own existence, goals, and limitations.
* **Decision-making**: The process by which a system evaluates options and selects a course of action.
* **Learning**: The ability of a system to adapt and improve its performance over time based on experience and feedback.

## Key Components of Agentic AI
The key components of Agentic AI work together to enable autonomous decision-making and action. These components include:

* **Autonomy**: Autonomous systems can operate independently, making decisions based on their own goals and objectives.
* **Self-awareness**: Self-aware systems can recognize their own strengths, weaknesses, and limitations, allowing them to make more informed decisions.
* **Decision-making**: Agentic AI systems use decision-making algorithms to evaluate options and select a course of action.
* **Learning**: Learning algorithms enable Agentic AI systems to adapt and improve their performance over time, based on experience and feedback.

## Real-World Applications of Agentic AI
Agentic AI has numerous real-world applications across various industries, including:

* **Robotics**: Agentic AI can be used to create autonomous robots that can navigate and interact with their environment.
* **Healthcare**: Agentic AI can be used to develop personalized healthcare systems that can monitor and respond to individual patient needs.
* **Smart Homes**: Agentic AI can be used to create smart home systems that can learn and adapt to the preferences and habits of their occupants.
* **Finance**: Agentic AI can be used to develop autonomous trading systems that can make decisions based on market analysis and trends.

## Technical Implementation Overview
The technical implementation of Agentic AI involves a range of techniques, including:

* **Machine learning**: Machine learning algorithms are used to enable Agentic AI systems to learn and adapt over time.
* **Natural language processing**: Natural language processing techniques are used to enable Agentic AI systems to understand and respond to human language.
* **Computer vision**: Computer vision techniques are used to enable Agentic AI systems to perceive and interpret visual data.
* **Software frameworks**: Software frameworks, such as cognitive architectures and agent-based modeling, are used to integrate and manage the various components of Agentic AI systems.

## Ethical Considerations and Future Outlook
As Agentic AI continues to evolve and become more widespread, there are several ethical considerations that must be addressed, including:

* **Accountability**: Who is responsible for the actions of an autonomous Agentic AI system?
* **Transparency**: How can we ensure that Agentic AI systems are transparent and explainable in their decision-making processes?
* **Bias**: How can we prevent Agentic AI systems from perpetuating existing biases and inequalities?
The future of Agentic AI holds much promise, with potential applications in areas such as space exploration, environmental sustainability, and social welfare. However, it is crucial that we prioritize ethical considerations and ensure that Agentic AI is developed and used responsibly.
TERMINATE

---

In [9]:
# ─────────────────────────────────────────────────────────────────
# Extract and Display the Final Refined Draft
# The last message from the Content Creator is the final draft
# ─────────────────────────────────────────────────────────────────

final_draft = None
for message in reversed(chat_result.chat_history):
    name = message.get("name", message.get("role", ""))
    if "Content_Creator" in name or "assistant" in name.lower():
        final_draft = message.get("content", "").replace("TERMINATE", "").strip()
        break

display(Markdown("# ✅ Final Refined Draft"))
display(Markdown("---"))
if final_draft:
    display(Markdown(final_draft))
else:
    display(Markdown("*Could not extract final draft — see conversation log above.*"))

# ✅ Final Refined Draft

---

# Introduction to Agentic AI
Agentic AI refers to a type of artificial intelligence that enables systems to act autonomously, making decisions and taking actions based on their own goals and objectives. This concept is rooted in the idea of agency, where an entity has the capacity to make choices and influence its environment.

## Definition and Key Concepts
Agentic AI combines various disciplines, including artificial intelligence, cognitive science, and philosophy, to create systems that can perceive, reason, and act upon their environment. Key concepts in Agentic AI include:

* **Autonomy**: The ability of a system to operate independently, making decisions without human intervention.
* **Self-awareness**: The capacity of a system to have a sense of its own existence, goals, and limitations.
* **Decision-making**: The process by which a system evaluates options and selects a course of action.
* **Learning**: The ability of a system to adapt and improve its performance over time based on experience and feedback.

## Key Components of Agentic AI
The key components of Agentic AI work together to enable autonomous decision-making and action. These components include:

* **Autonomy**: Autonomous systems can operate independently, making decisions based on their own goals and objectives.
* **Self-awareness**: Self-aware systems can recognize their own strengths, weaknesses, and limitations, allowing them to make more informed decisions.
* **Decision-making**: Agentic AI systems use decision-making algorithms to evaluate options and select a course of action.
* **Learning**: Learning algorithms enable Agentic AI systems to adapt and improve their performance over time, based on experience and feedback.

## Real-World Applications of Agentic AI
Agentic AI has numerous real-world applications across various industries, including:

* **Robotics**: Agentic AI can be used to create autonomous robots that can navigate and interact with their environment.
* **Healthcare**: Agentic AI can be used to develop personalized healthcare systems that can monitor and respond to individual patient needs.
* **Smart Homes**: Agentic AI can be used to create smart home systems that can learn and adapt to the preferences and habits of their occupants.
* **Finance**: Agentic AI can be used to develop autonomous trading systems that can make decisions based on market analysis and trends.

## Technical Implementation Overview
The technical implementation of Agentic AI involves a range of techniques, including:

* **Machine learning**: Machine learning algorithms are used to enable Agentic AI systems to learn and adapt over time.
* **Natural language processing**: Natural language processing techniques are used to enable Agentic AI systems to understand and respond to human language.
* **Computer vision**: Computer vision techniques are used to enable Agentic AI systems to perceive and interpret visual data.
* **Software frameworks**: Software frameworks, such as cognitive architectures and agent-based modeling, are used to integrate and manage the various components of Agentic AI systems.

## Ethical Considerations and Future Outlook
As Agentic AI continues to evolve and become more widespread, there are several ethical considerations that must be addressed, including:

* **Accountability**: Who is responsible for the actions of an autonomous Agentic AI system?
* **Transparency**: How can we ensure that Agentic AI systems are transparent and explainable in their decision-making processes?
* **Bias**: How can we prevent Agentic AI systems from perpetuating existing biases and inequalities?
The future of Agentic AI holds much promise, with potential applications in areas such as space exploration, environmental sustainability, and social welfare. However, it is crucial that we prioritize ethical considerations and ensure that Agentic AI is developed and used responsibly.

In [10]:
# ─────────────────────────────────────────────────────────────────
# Conversation Statistics
# ─────────────────────────────────────────────────────────────────

total_turns = len(chat_result.chat_history)
creator_turns = sum(1 for m in chat_result.chat_history
                    if "Content_Creator" in m.get("name", m.get("role", "")))
critic_turns  = sum(1 for m in chat_result.chat_history
                    if "Content_Critic" in m.get("name", m.get("role", "")))

display(Markdown("## 📊 Session Statistics"))
display(Markdown(f"""
| Metric | Value |
|--------|-------|
| Total turns | {total_turns} |
| Content Creator turns | {creator_turns} |
| Content Critic turns | {critic_turns} |
| Framework | AutoGen |
| Model | llama-3.3-70b-versatile (via Groq) |
"""))

## 📊 Session Statistics


| Metric | Value |
|--------|-------|
| Total turns | 2 |
| Content Creator turns | 1 |
| Content Critic turns | 1 |
| Framework | AutoGen |
| Model | llama-3.3-70b-versatile (via Groq) |
